In [65]:
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, List,Literal,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain_google_genai import ChatGoogleGenerativeAI
import google.genai as genai
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage,HumanMessage,BaseMessage,AIMessage
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
import os

In [ ]:
client = genai.Client(
    vertexai=True, 
    project=os.getenv('GEMINI_API_KEY'), 
    location='us-central1'
)

In [67]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [68]:
def chat_node(state:ChatState):

    #take user query from the state
    messages = state['messages']
    
    # Convert ALL messages to text format for Google client
    # This maintains conversation context
    conversation_text = ""
    for msg in messages:
        if isinstance(msg, HumanMessage):
            conversation_text += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            conversation_text += f"Assistant: {msg.content}\n"
    
    # Add current context instruction
    full_prompt = f"""You are a helpful assistant. Here's our conversation history:

{conversation_text}

Please respond to the latest user message while keeping the conversation context in mind."""
    
    #send to llm with full context
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=full_prompt
    ).text

    #response store state - return as AIMessage to maintain message format
    return {'messages':[AIMessage(content=response)]}


In [69]:
checkpointer  = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

In [70]:
workflow = graph.compile(checkpointer=checkpointer)

In [71]:
#Displaying the graph
print(workflow.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| chat_node |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [72]:
# initial_state = {'messages':[HumanMessage(content="What is LangGraph?")]}
# ans = workflow.invoke(initial_state)['messages'][-1].content

In [73]:
# print(ans)

In [75]:
thread_id = '1'
while True:
    user_message = input("Type Here  : ")

    print("user : ",user_message)

    if user_message.strip().lower() in ['exit','quit','stop','bye']:
        print("Exiting the chat. Goodbye!")
        break

    config = {'configurable': {'thread_id': thread_id}}

    response = workflow.invoke({'messages':[HumanMessage(content=user_message)]},config=config)['messages'][-1].content

    print("Chatbot : ",response)


user :  hi
Chatbot :  Hi Siddhi! How can I help you today?

user :  2 + 2
Chatbot :  2 + 2 = 4. Is there anything else I can help you with, Siddhi?

user :  +2
Chatbot :  Okay, Siddhi. Do you want me to add 2 to the previous answer of 4? That would be 6. Or are you starting a new calculation? Let me know!

user :  add to previous
Chatbot :  Okay, Siddhi. Adding 2 to the previous answer of 4, which then became 6, would make the new answer 8. Is that correct?

user :  yes
Chatbot :  Great! So the current answer is 8. What would you like to do next, Siddhi?

user :  bye
Exiting the chat. Goodbye!


In [76]:
workflow.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='a07e8d9e-38de-4ae9-8fa4-f092e7facfce'), AIMessage(content='Hi there! How can I help you today?\n', additional_kwargs={}, response_metadata={}, id='9d04a95b-35e1-4d2e-94b3-162477492167', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='my name is siddhi', additional_kwargs={}, response_metadata={}, id='1b1e2631-0d96-42a3-b5fc-2238afe61897'), AIMessage(content="Hi Siddhi! It's nice to meet you. How can I help you today?\n", additional_kwargs={}, response_metadata={}, id='0e75d382-3da9-4d7e-9d3f-84ff6e55d959', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='wha is my name', additional_kwargs={}, response_metadata={}, id='fda8197a-d466-4d30-8bf3-39a518c5649d'), AIMessage(content='Your name is Siddhi.\n', additional_kwargs={}, response_metadata={}, id='84cde4af-7aa9-4aef-b648-6d65fb077075', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='ok', ad